# FD001 Advanced Machine Learning for Remaining Useful Life Prediction

**Student:** Ivan Da Silva  
**Course:** AAI 501 — Introduction to Artificial Intelligence  
**Component:** Advanced ML, hyperparameter tuning, feature importance, and model comparison

This notebook extends notebook 01's validated FD001 preparation and notebook 02's baseline regression experiment. It trains and tunes an XGBoost regressor for Remaining Useful Life (RUL) prediction using the centralized rules in `docs/FD001_EXPERIMENT_CONTRACT.md`.

**Scope caveat.** FD001 has 100 training and 100 test trajectories, one sea-level operating condition, and one HPC-degradation fault mode. The experiment isolates model behavior in a controlled case; it does not establish generalization to the multiple operating conditions or fan-degradation modes represented by FD002–FD004.

The workflow follows the Module 4 machine-learning lifecycle: define the problem, preserve a valid holdout set, tune complexity through cross-validation, compare models under identical conditions, and interpret feature importance. All training and validation splits are made by engine so observations from the same engine cannot appear in both partitions.

## 1. Experimental Question and Success Criteria

**Question:** Can a tuned XGBoost regressor improve FD001 RUL prediction over Linear Regression, Random Forest, and Gradient Boosting when every model uses the same features, capped target, engine-level split, and official test construction?

The primary metric is RMSE because large RUL errors receive greater penalties. MAE is included because it is easy to interpret as the average absolute error in cycles, and R² describes the proportion of target variation explained by the model.

To make the comparison fair, this notebook follows Mina's baseline choices:

- FD001 only;
- RUL capped at 125 cycles;
- the same 18 non-constant setting and sensor features;
- 80/20 training-validation split by engine with `random_state=42`;
- official test evaluation at each engine's final observed cycle.

The official test set is never used for hyperparameter selection.

## 2. Setup and Reproducibility

Run the next cell first. If XGBoost is missing in Google Colab, uncomment the installation line, run the cell once, and then comment it again before committing the notebook.

In [ ]:
# Uncomment only if XGBoost is not installed in the notebook environment.
# %pip install -q xgboost

from pathlib import Path
import platform
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

RANDOM_STATE = 42
RUL_CAP = 125

np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"XGBoost: {xgboost.__version__}")

## 3. Locate the Repository and Load FD001

The path logic supports running the notebook from the repository root, the `notebooks` folder, or a standard `/content/intelligent-predictive-maintenance` Colab clone. It consumes committed project data and never downloads a replacement from a mutable branch.

Before running notebook 03, run notebooks 01 and 02 in order. Notebook 01 owns the canonical processed CSV, and notebook 02 saves the shared split and baseline artifacts.

In [ ]:
def find_repo_root():
    # Return the project root containing README.md and datasets/.
    starts = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/intelligent-predictive-maintenance"),
    ]

    for start in starts:
        candidates = [start, *start.parents]
        for candidate in candidates:
            if (
                (candidate / "README.md").exists()
                and (candidate / "datasets").exists()
            ):
                return candidate.resolve()

    raise FileNotFoundError(
        "Repository root not found. Open this notebook from the cloned "
        "intelligent-predictive-maintenance repository."
    )


REPO_ROOT = find_repo_root()
PROCESSED_PATH = (
    REPO_ROOT / "datasets" / "processed" / "fd001_train_with_rul.csv"
)
RAW_DIR = REPO_ROOT / "datasets" / "raw" / "CMAPSSData"
TEST_PATH = RAW_DIR / "test_FD001.txt"
TEST_RUL_PATH = RAW_DIR / "RUL_FD001.txt"

required_paths = [PROCESSED_PATH, TEST_PATH, TEST_RUL_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        f"Required project files are missing:\n{missing_text}"
    )

print(f"Repository root: {REPO_ROOT}")
print(f"Processed training data: {PROCESSED_PATH}")
print(f"Raw test data: {TEST_PATH}")

In [ ]:
column_names = (
    ["engine_id", "cycle", "setting1", "setting2", "setting3"]
    + [f"sensor{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(PROCESSED_PATH)
test_df = pd.read_csv(TEST_PATH, sep=r"\s+", header=None, names=column_names)
test_rul_df = pd.read_csv(
    TEST_RUL_PATH,
    sep=r"\s+",
    header=None,
    names=["RUL"],
)
test_rul_df["engine_id"] = np.arange(1, len(test_rul_df) + 1)

print(f"Training shape: {train_df.shape}")
print(f"Test trajectory shape: {test_df.shape}")
print(f"Official test labels: {test_rul_df.shape}")
train_df.head()

## 4. Validate the Data Handoff and Shared Contract

The following checks verify notebook 01's canonical processed training data. The canonical file must contain exactly the 26 labeled raw columns plus `max_cycle` and `RUL`; EDA-only fields such as `RUL_bin` are prohibited. Notebook 03 then recreates notebook 02's documented feature and split choices for an executable fair comparison.

In [ ]:
required_train_columns = set(column_names + ["max_cycle", "RUL"])
missing_columns = required_train_columns.difference(train_df.columns)

assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
assert len(train_df) == 20631, "Unexpected FD001 training row count."
assert train_df["engine_id"].nunique() == 100, "Expected 100 engines."
assert not train_df.duplicated(["engine_id", "cycle"]).any()
assert not train_df[list(required_train_columns)].isna().any().any()
assert (train_df["RUL"] >= 0).all()
assert train_df.groupby("engine_id")["RUL"].min().eq(0).all()
assert len(test_rul_df) == test_df["engine_id"].nunique() == 100

eda_only_columns = sorted(set(train_df.columns) - required_train_columns)
print("All data-handoff checks passed.")
print(f"EDA-only columns excluded from modeling: {eda_only_columns}")

## 5. Target, Features, and Engine-Level Holdout Split

RUL is capped at 125 cycles to match the baseline experiment. This piecewise-linear assumption treats early healthy-life observations as equivalent once they are more than 125 cycles from failure.

The split is performed on engine IDs, not individual rows. Consecutive observations from one engine are highly related; a row-level split would leak near-duplicate engine states across partitions.

In [ ]:
constant_sensors = [
    "sensor1",
    "sensor5",
    "sensor10",
    "sensor16",
    "sensor18",
    "sensor19",
]
setting_columns = ["setting1", "setting2", "setting3"]
sensor_columns = [
    column
    for column in column_names
    if column.startswith("sensor") and column not in constant_sensors
]
feature_columns = setting_columns + sensor_columns

train_df = train_df.copy()
train_df["RUL_capped"] = train_df["RUL"].clip(upper=RUL_CAP)

engine_ids = np.sort(train_df["engine_id"].unique())
train_ids, validation_ids = train_test_split(
    engine_ids,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_mask = train_df["engine_id"].isin(train_ids)
validation_mask = train_df["engine_id"].isin(validation_ids)

development_df = train_df.loc[train_mask].copy()
validation_df = train_df.loc[validation_mask].copy()

X_train = development_df[feature_columns]
y_train = development_df["RUL_capped"]
train_groups = development_df["engine_id"]

X_validation = validation_df[feature_columns]
y_validation = validation_df["RUL_capped"]

assert set(train_ids).isdisjoint(validation_ids)
assert len(feature_columns) == 18

print(f"Training engines: {len(train_ids)}; rows: {len(X_train):,}")
print(
    f"Validation engines: {len(validation_ids)}; "
    f"rows: {len(X_validation):,}"
)
print(f"Predictor count: {len(feature_columns)}")
print(feature_columns)

In [ ]:
test_last_cycle = (
    test_df.sort_values(["engine_id", "cycle"])
    .groupby("engine_id", as_index=False)
    .tail(1)
    .reset_index(drop=True)
    .merge(test_rul_df, on="engine_id", how="left", validate="one_to_one")
)
test_last_cycle["RUL_capped"] = test_last_cycle["RUL"].clip(upper=RUL_CAP)

X_test = test_last_cycle[feature_columns]
y_test = test_last_cycle["RUL_capped"]
y_test_uncapped = test_last_cycle["RUL"]

assert len(X_test) == 100
assert not test_last_cycle["RUL"].isna().any()

test_last_cycle[["engine_id", "cycle", "RUL", "RUL_capped"]].head()

## 6. Reproduce the Traditional ML Baselines

The three baseline models are reconstructed with Mina's settings. This makes the comparison executable inside one notebook even though notebook 02 did not commit its saved model files. Standardization is fit only on the training partition.

In [ ]:
def regression_metrics(y_true, predictions):
    # Return common RUL regression metrics.
    return {
        "RMSE": mean_squared_error(y_true, predictions) ** 0.5,
        "MAE": mean_absolute_error(y_true, predictions),
        "R2": r2_score(y_true, predictions),
    }


def evaluate_partitions(model, X_val, y_val, X_official, y_official):
    # Evaluate one fitted model on validation and official test data.
    validation_predictions = model.predict(X_val)
    test_predictions = model.predict(X_official)
    return (
        regression_metrics(y_val, validation_predictions),
        regression_metrics(y_official, test_predictions),
        validation_predictions,
        test_predictions,
    )


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

baseline_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
    ),
}

baseline_results = []
baseline_test_predictions = {}

for model_name, model in baseline_models.items():
    model.fit(X_train_scaled, y_train)
    val_metrics, test_metrics, _, test_predictions = evaluate_partitions(
        model,
        X_validation_scaled,
        y_validation,
        X_test_scaled,
        y_test,
    )
    baseline_test_predictions[model_name] = test_predictions
    baseline_results.append(
        {
            "Model": model_name,
            "Validation_RMSE": val_metrics["RMSE"],
            "Validation_MAE": val_metrics["MAE"],
            "Validation_R2": val_metrics["R2"],
            "Test_RMSE": test_metrics["RMSE"],
            "Test_MAE": test_metrics["MAE"],
            "Test_R2": test_metrics["R2"],
        }
    )

baseline_results_df = pd.DataFrame(baseline_results).sort_values("Test_RMSE")
baseline_results_df.round(3)

## 7. Initial XGBoost Regressor

XGBoost builds trees sequentially. Each new tree attempts to reduce errors remaining from the earlier ensemble. The initial model uses a moderate tree depth, learning rate, and subsampling to establish an untuned advanced-ML result.

Tree models do not require standardized feature scales, so XGBoost uses the original feature values.

In [ ]:
initial_xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    max_depth=3,
    learning_rate=0.10,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)
initial_xgb.fit(X_train, y_train)

(
    initial_validation_metrics,
    initial_test_metrics,
    initial_validation_predictions,
    initial_test_predictions,
) = evaluate_partitions(
    initial_xgb,
    X_validation,
    y_validation,
    X_test,
    y_test,
)

pd.DataFrame(
    [
        {"Partition": "Validation", **initial_validation_metrics},
        {"Partition": "Official test", **initial_test_metrics},
    ]
).round(3)

## 8. Group-Aware Hyperparameter Tuning

The grid search varies three concepts covered in the tree-based-model material:

- `max_depth`: model complexity and overfitting risk;
- `learning_rate`: the contribution of each new tree;
- `n_estimators`: the number of sequential trees.

Three-fold `GroupKFold` keeps every engine wholly within one fold. The search minimizes RMSE using only the 80 development engines. The 20 validation engines and 100 official test engines remain outside the tuning process.

In [ ]:
tuning_model = XGBRegressor(
    objective="reg:squarederror",
    subsample=0.80,
    colsample_bytree=0.80,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)

parameter_grid = {
    "n_estimators": [200, 400],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.10],
}

group_cv = GroupKFold(n_splits=3)
grid_search = GridSearchCV(
    estimator=tuning_model,
    param_grid=parameter_grid,
    scoring="neg_root_mean_squared_error",
    cv=group_cv,
    refit=True,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train, groups=train_groups)

print(f"Best cross-validation RMSE: {-grid_search.best_score_:.3f}")
print("Best parameters:")
print(grid_search.best_params_)

In [ ]:
cv_results_df = pd.DataFrame(grid_search.cv_results_)
cv_summary = (
    cv_results_df[
        [
            "param_n_estimators",
            "param_max_depth",
            "param_learning_rate",
            "mean_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .assign(
        Mean_Train_RMSE=lambda frame: -frame["mean_train_score"],
        Mean_CV_RMSE=lambda frame: -frame["mean_test_score"],
    )
    .sort_values("rank_test_score")
)

cv_summary[
    [
        "rank_test_score",
        "param_n_estimators",
        "param_max_depth",
        "param_learning_rate",
        "Mean_Train_RMSE",
        "Mean_CV_RMSE",
        "std_test_score",
    ]
].head(10).round(3)

## 9. Tuned XGBoost Evaluation and Model Comparison

The selected model is first evaluated on the untouched 20-engine validation partition and then on the official test endpoints. The table combines the traditional baselines, initial XGBoost, and tuned XGBoost under the same capped-RUL experiment.

In [ ]:
tuned_xgb = grid_search.best_estimator_
(
    tuned_validation_metrics,
    tuned_test_metrics,
    tuned_validation_predictions,
    tuned_test_predictions,
) = evaluate_partitions(
    tuned_xgb,
    X_validation,
    y_validation,
    X_test,
    y_test,
)

xgb_results = [
    {
        "Model": "Initial XGBoost",
        "Validation_RMSE": initial_validation_metrics["RMSE"],
        "Validation_MAE": initial_validation_metrics["MAE"],
        "Validation_R2": initial_validation_metrics["R2"],
        "Test_RMSE": initial_test_metrics["RMSE"],
        "Test_MAE": initial_test_metrics["MAE"],
        "Test_R2": initial_test_metrics["R2"],
    },
    {
        "Model": "Tuned XGBoost",
        "Validation_RMSE": tuned_validation_metrics["RMSE"],
        "Validation_MAE": tuned_validation_metrics["MAE"],
        "Validation_R2": tuned_validation_metrics["R2"],
        "Test_RMSE": tuned_test_metrics["RMSE"],
        "Test_MAE": tuned_test_metrics["MAE"],
        "Test_R2": tuned_test_metrics["R2"],
    },
]

comparison_df = (
    pd.concat(
        [baseline_results_df, pd.DataFrame(xgb_results)],
        ignore_index=True,
    )
    .sort_values("Test_RMSE")
    .reset_index(drop=True)
)

comparison_df.round(3)

In [ ]:
best_baseline_rmse = baseline_results_df["Test_RMSE"].min()
tuned_rmse = tuned_test_metrics["RMSE"]
rmse_change = best_baseline_rmse - tuned_rmse
percent_change = 100 * rmse_change / best_baseline_rmse

if rmse_change > 0:
    comparison_statement = (
        f"Tuned XGBoost reduced test RMSE by {rmse_change:.3f} cycles "
        f"({percent_change:.2f}%) relative to the strongest baseline."
    )
elif rmse_change < 0:
    comparison_statement = (
        f"Tuned XGBoost test RMSE was {-rmse_change:.3f} cycles higher "
        f"than the strongest baseline."
    )
else:
    comparison_statement = "Tuned XGBoost tied the strongest baseline."

print(comparison_statement)

plot_df = comparison_df.melt(
    id_vars="Model",
    value_vars=["Validation_RMSE", "Test_RMSE"],
    var_name="Partition",
    value_name="RMSE",
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Model", y="RMSE", hue="Partition")
plt.title("Fair Model Comparison: Capped RUL RMSE")
plt.xlabel("")
plt.ylabel("RMSE (cycles)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 10. Prediction Errors Across RUL Ranges

An overall average can hide where a model performs poorly. The next analysis separates official test engines by their capped RUL. For maintenance planning, errors near end-of-life are especially important because those engines may require urgent attention.

In [ ]:
prediction_df = test_last_cycle[
    ["engine_id", "cycle", "RUL", "RUL_capped"]
].copy()
prediction_df["Predicted_RUL"] = tuned_test_predictions
prediction_df["Residual"] = (
    prediction_df["RUL_capped"] - prediction_df["Predicted_RUL"]
)
prediction_df["Absolute_Error"] = prediction_df["Residual"].abs()
prediction_df["RUL_Range"] = pd.cut(
    prediction_df["RUL_capped"],
    bins=[-1, 25, 50, 100, RUL_CAP],
    labels=["0–25", "26–50", "51–100", "101–125"],
)

range_results = (
    prediction_df.groupby("RUL_Range", observed=False)
    .agg(
        Engines=("engine_id", "count"),
        MAE=("Absolute_Error", "mean"),
        Median_AE=("Absolute_Error", "median"),
        Mean_Residual=("Residual", "mean"),
    )
    .reset_index()
)
range_results.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

limit = max(
    prediction_df["RUL_capped"].max(),
    prediction_df["Predicted_RUL"].max(),
) + 5
axes[0].scatter(
    prediction_df["RUL_capped"],
    prediction_df["Predicted_RUL"],
    alpha=0.70,
    edgecolor="black",
    linewidth=0.3,
)
axes[0].plot([0, limit], [0, limit], "r--", linewidth=1)
axes[0].set_title("Tuned XGBoost: Actual vs. Predicted RUL")
axes[0].set_xlabel("Actual capped RUL")
axes[0].set_ylabel("Predicted RUL")

sns.scatterplot(
    data=prediction_df,
    x="Predicted_RUL",
    y="Residual",
    hue="RUL_Range",
    ax=axes[1],
)
axes[1].axhline(0, color="red", linestyle="--", linewidth=1)
axes[1].set_title("Residuals by Predicted RUL")
axes[1].set_xlabel("Predicted RUL")
axes[1].set_ylabel("Actual − predicted")

plt.tight_layout()
plt.show()

## 11. Feature Importance and Interpretation

XGBoost's built-in importance describes how the fitted ensemble used predictors. Permutation importance provides a complementary check by measuring how much validation RMSE worsens when one feature is shuffled. Neither measure proves that a sensor causes degradation.

In [ ]:
built_in_importance = pd.DataFrame(
    {
        "Feature": feature_columns,
        "XGBoost_Importance": tuned_xgb.feature_importances_,
    }
).sort_values("XGBoost_Importance", ascending=False)

permutation = permutation_importance(
    tuned_xgb,
    X_validation,
    y_validation,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
permutation_df = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Permutation_Importance": permutation.importances_mean,
        "Permutation_SD": permutation.importances_std,
    }
).sort_values("Permutation_Importance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top_builtin = built_in_importance.head(12).sort_values(
    "XGBoost_Importance"
)
axes[0].barh(top_builtin["Feature"], top_builtin["XGBoost_Importance"])
axes[0].set_title("XGBoost Built-in Feature Importance")
axes[0].set_xlabel("Importance")

top_permutation = permutation_df.head(12).sort_values(
    "Permutation_Importance"
)
axes[1].barh(
    top_permutation["Feature"],
    top_permutation["Permutation_Importance"],
    xerr=top_permutation["Permutation_SD"],
)
axes[1].set_title("Validation Permutation Importance")
axes[1].set_xlabel("Increase in RMSE after shuffling")

plt.tight_layout()
plt.show()

permutation_df.head(10).round(4)

## 12. Capped and Uncapped Test Metrics

The main comparison uses capped RUL because that matches notebook 02. The following secondary calculation evaluates the same predictions against uncapped official RUL labels. These values answer a different question and must not be mixed with the capped comparison.

In [ ]:
metric_definition_df = pd.DataFrame(
    [
        {
            "Target definition": "Capped at 125",
            **regression_metrics(y_test, tuned_test_predictions),
        },
        {
            "Target definition": "Official uncapped RUL",
            **regression_metrics(y_test_uncapped, tuned_test_predictions),
        },
    ]
)
metric_definition_df.round(3)

## 13. Save Reproducible Results

This cell saves small, reviewable result artifacts and the tuned model under the repository. Review these files with `git status` before deciding which ones belong in the pull request.

In [ ]:
reports_dir = REPO_ROOT / "reports"
models_dir = REPO_ROOT / "models"
reports_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)

comparison_path = reports_dir / "advanced_ml_model_comparison.csv"
predictions_path = reports_dir / "advanced_ml_test_predictions.csv"
cv_path = reports_dir / "advanced_ml_cv_results.csv"
split_path = reports_dir / "advanced_ml_engine_split.csv"
model_path = models_dir / "tuned_xgboost_fd001.json"

comparison_df.to_csv(comparison_path, index=False)
prediction_df.to_csv(predictions_path, index=False)
cv_summary.to_csv(cv_path, index=False)
pd.DataFrame(
    {
        "engine_id": np.concatenate([train_ids, validation_ids]),
        "partition": (
            ["train"] * len(train_ids)
            + ["validation"] * len(validation_ids)
        ),
    }
).to_csv(split_path, index=False)
tuned_xgb.save_model(model_path)
joblib.dump(scaler, models_dir / "baseline_scaler_for_comparison.pkl")

print("Saved:")
for path in [comparison_path, predictions_path, cv_path, split_path, model_path]:
    print(f"- {path.relative_to(REPO_ROOT)}")

## 14. Conclusion and Limitations

After running all cells, use the generated comparison statement, model table, residual plots, RUL-range table, and importance charts to answer whether tuned XGBoost improved upon the strongest baseline and by how many RMSE cycles or percent.

Important limitations:

- Only FD001 is evaluated: one simulated sea-level condition and one HPC-degradation fault mode.
- Results cannot be generalized to FD002–FD004 without new experiments.
- Capping RUL at 125 changes the prediction target and must be disclosed.
- Tuning results depend on the selected parameter grid and grouped folds.
- Feature importance describes model behavior, not causal sensor effects.
- Official test predictions are endpoint estimates, not a maintenance schedule.
- C-MAPSS does not supply maintenance costs, service duration, capacity, or safety thresholds.

The model outputs can support downstream decision-making by supplying predicted RUL and error estimates, but maintenance priorities require transparent risk thresholds and operational assumptions.

## References

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785

National Aeronautics and Space Administration. (n.d.). *C-MAPSS jet engine simulated data* [Data set]. NASA Open Data Portal. https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data

Scikit-learn developers. (n.d.). *GridSearchCV*. Scikit-learn. https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

XGBoost developers. (n.d.). *XGBoost Python package*. https://xgboost.readthedocs.io/

### AI Use Disclosure

I used ChatGPT/Codex to help organize the notebook, adapt the Module 4 XGBoost workflow from classification to RUL regression, review the experimental design, and improve code clarity. I ran the notebook, reviewed the outputs and visualizations, and made the final decisions about model interpretation and conclusions.